# Notebook 01 — Dataset Exploration and Preprocessing
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** What does our dataset look like, and are our DataLoaders correctly configured before we invest GPU time in training?

**Phase 1 rubric targets:**
- Visualize ≥5 sample images with labels (1 pt)
- Class distribution bar chart (1 pt)
- Functional train / val / test DataLoaders (1 pt)

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────
# Why first cell is always config: single source of truth for paths
# and hyperparameters. Any change here propagates to all cells below.
import logging
import sys
from pathlib import Path

# Add project root to sys.path so src.* imports resolve in both
# PyCharm (local) and Colab (after Drive mount)
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)

from src.config import (
    DEVICE, EXPERIMENTS_DIR, NUM_CLASSES, SEED,
    TEST_DIR, TRAIN_DIR,
)
from src.utils import set_seed

set_seed(SEED)
print(f'Device      : {DEVICE}')
print(f'Train dir   : {TRAIN_DIR}  exists={TRAIN_DIR.exists()}')
print(f'Test dir    : {TEST_DIR}   exists={TEST_DIR.exists()}')
print(f'Num classes : {NUM_CLASSES}')

## 1. Dataset Structure Audit

Before loading anything into memory, we audit the raw folder structure.
This catches missing classes or corrupt directories before training.

In [ ]:
# ── Cell 2: Dataset structure audit ────────────────────────────
# Why audit before loading: ImageFolder silently skips malformed
# subdirectories. Explicit counting confirms the expected 50 classes.
import os

train_classes = sorted(os.listdir(TRAIN_DIR))
test_classes  = sorted(os.listdir(TEST_DIR))

print(f'Train classes found : {len(train_classes)}')
print(f'Test  classes found : {len(test_classes)}')
print(f'Classes match       : {train_classes == test_classes}')
print(f'First 5 classes     : {train_classes[:5]}')

# Count images per class — reveals class imbalance before training
train_counts = {
    cls: len(list((TRAIN_DIR / cls).glob('*')))
    for cls in train_classes
}
total_train = sum(train_counts.values())
print(f'Total train images  : {total_train}')
print(f'Avg per class       : {total_train / len(train_classes):.1f}')

## 2. Class Distribution — BI Visualization

**Storytelling question:** Is the dataset balanced across all 50 landmark classes?

Class imbalance forces the model to be biased toward majority classes.
If detected here, we address it via weighted sampling or loss weighting — before wasting compute on a biased model.

In [ ]:
# ── Cell 3: BI class distribution plot ─────────────────────────
# UCB palette applied — color encodes information (not decoration):
# UCB Gold = majority classes (potential bias source)
# UCB Blue = standard classes
import matplotlib.pyplot as plt
import numpy as np

UCB_BLUE      = '#003262'
UCB_GOLD      = '#FDB515'
UCB_DARK_GOLD = '#C4820E'

names  = list(train_counts.keys())
counts = list(train_counts.values())
mean_count = np.mean(counts)

colors = [UCB_GOLD if c > mean_count * 1.5 else UCB_BLUE for c in counts]

fig, ax = plt.subplots(figsize=(22, 5), facecolor='white')
bars = ax.bar(range(len(names)), counts, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(mean_count, color=UCB_DARK_GOLD, linestyle='--', linewidth=1.2,
           label=f'Mean: {mean_count:.0f} images/class')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('_', ' ') for n in names],
                   rotation=90, fontsize=6)
ax.set_title('Landmark Dataset — Class Distribution\n'
             'Gold bars = majority classes (>1.5x mean) — potential bias source',
             fontsize=12, fontweight='bold', color=UCB_BLUE)
ax.set_xlabel('Landmark Class', fontsize=9, color=UCB_BLUE)
ax.set_ylabel('Number of Images', fontsize=9, color=UCB_BLUE)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

out = EXPERIMENTS_DIR / 'class_distribution.png'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')

## 3. Sample Images with Labels

Visual inspection of raw images confirms:
- Images load correctly from the dataset path
- Labels map to recognizable landmark names
- No obvious data corruption

In [ ]:
# ── Cell 4: Sample images visualization ─────────────────────────
# Why visual inspection before DataLoader: ImageFolder may load
# images correctly by shape but with wrong label mapping.
# Human review catches mismatches that metrics cannot.
import random
from torchvision.datasets import ImageFolder

raw_ds = ImageFolder(str(TRAIN_DIR))
sample_idx = random.sample(range(len(raw_ds)), 8)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), facecolor='white')
axes = axes.flatten()

for ax, idx in zip(axes, sample_idx):
    img, label = raw_ds[idx]
    ax.imshow(img)
    ax.set_title(
        raw_ds.classes[label].replace('_', ' '),
        fontsize=8, color=UCB_BLUE, fontweight='bold'
    )
    ax.axis('off')

fig.suptitle('Sample Landmark Images with Ground Truth Labels',
             fontsize=12, fontweight='bold', color=UCB_BLUE)
plt.tight_layout()

out = EXPERIMENTS_DIR / 'sample_images.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')

## 4. DataLoaders — Build and Verify

We build three DataLoaders:
- **Train**: shuffle=True + augmentation active
- **Val**: shuffle=False + no augmentation (reproducible metrics)
- **Test**: shuffle=False + no augmentation (held-out evaluation)

In [ ]:
# ── Cell 5: DataLoaders ─────────────────────────────────────────
# Why verify_dataloaders before any training:
# Shape mismatches and normalization errors are silent —
# the model trains but learns nothing useful.
# Catching them here costs seconds, not hours of GPU time.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)

## 5. Augmentation Preview

Showing the same image before and after augmentation confirms:
- Augmentation is active on training samples
- Transforms are reasonable (not destroying landmark structure)
- Normalization shifts pixel values to the expected ImageNet range

In [ ]:
# ── Cell 6: Augmentation preview ───────────────────────────────
# Why show augmented vs original side by side:
# Augmentation that is too aggressive (e.g. extreme rotation on
# symmetric landmarks) can confuse the model more than help it.
# Visual confirmation prevents silent accuracy degradation.
from src.data import get_transforms
from PIL import Image
import torch

# Load one raw image for comparison
sample_class = class_names[0]
sample_path  = next((TRAIN_DIR / sample_class).glob('*'))
raw_img      = Image.open(sample_path).convert('RGB')

transform_plain = get_transforms(augment=False)
transform_aug   = get_transforms(augment=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='white')
axes[0].imshow(raw_img)
axes[0].set_title('Original', color=UCB_BLUE, fontweight='bold')

for i, ax in enumerate(axes[1:], 1):
    aug_tensor = transform_aug(raw_img)
    # Denormalize for display: reverse ImageNet normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = (aug_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    ax.set_title(f'Augmented #{i}', color=UCB_DARK_GOLD, fontweight='bold')

for ax in axes:
    ax.axis('off')

fig.suptitle(f'Augmentation Preview — {sample_class.replace("_", " ")}',
             fontsize=11, fontweight='bold', color=UCB_BLUE)
plt.tight_layout()
out = EXPERIMENTS_DIR / 'augmentation_preview.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')

## Phase 1 — Summary and Checklist

| Check | Status |
|---|---|
| 50 classes detected | ✅ |
| Class distribution plotted | ✅ |
| ≥5 sample images displayed | ✅ |
| DataLoaders: batch shape [B, 3, 224, 224] | ✅ |
| Augmentation preview confirmed | ✅ |
| All artifacts saved to experiments/ | ✅ |

**Next step:** `02_cnn_from_scratch.ipynb` — Phase 2 training on Colab T4.